<a href="https://colab.research.google.com/github/internshipinabook/nlp-internship-in-a-book/blob/main/notebooks/week10_fairness_STARTER.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ── Colab setup (skip if running locally) ──────────────────────
import os
if not os.path.exists('requirements.txt'):
    !git clone https://github.com/internshipinabook/nlp-internship-in-a-book.git book
    %cd book
    !pip install -r requirements.txt -q
    print('Colab setup complete ✅')
else:
    print('Running locally ✅')


# Week 10 — Fairness in Language Models (STARTER)
### The NLP Internship | LinguaAI Labs

This week: bias taxonomy, per-language audit, dialect bias measurement, mitigation, the fairness brief.

> ⏸️ **Pause and Predict**
> Before running the audit: which type of bias do you expect to be most severe — representation, measurement, or aggregation? Why?

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import warnings; warnings.filterwarnings("ignore")
np.random.seed(42)
from sklearn.metrics import f1_score, classification_report
from sklearn.model_selection import train_test_split
print("Setup complete ✅")
print("Note: Weeks 10-12 assume AfroXLMR is fine-tuned and saved in models/afro_xlmr/")
print("If not: run week9_fine_tuning_SOLUTION.ipynb first.")

## Part 1 — Three Types of NLP Bias

In [ ]:
# The bias taxonomy
bias_taxonomy = {
    "Representation bias": {
        "definition": "Some groups are underrepresented in training data",
        "example": "Pidgin tickets are 20% of corpus but near-zero in English NLP pre-training",
        "measurement": "Per-group OOV rate; per-group evaluation metrics",
    },
    "Measurement bias": {
        "definition": "Labels capture one group's experience better than another's",
        "example": "Abuse labels defined by English speakers — Pidgin abuse expressions inconsistently labelled",
        "measurement": "Inter-annotator agreement broken down by ticket language",
    },
    "Aggregation bias": {
        "definition": "One model used across groups with meaningfully different characteristics",
        "example": "Single classifier for English, Pidgin, Yorùbá — performance optimised for majority",
        "measurement": "Per-group performance gap vs aggregate performance",
    },
}
for bias_type, info in bias_taxonomy.items():
    print(f"\n{bias_type}:")
    for k, v in info.items():
        print(f"  {k}: {v[:80]}")

# 📚 Blodgett et al. (2020) "Language (Technology) is Power" — ACL 2020
# 📚 Bender et al. (2021) "Stochastic Parrots" — FAccT 2021
# 📚 Barocas, Hardt & Narayanan "Fairness and Machine Learning" — fairmlbook.org

## Part 2 — Performance Audit by Language Group

> 🧠 **What Will This Output?**
> Predict the English-Pidgin F1 gap before running. Is it closer to 5 points, 15 points, or 25 points?

In [ ]:
import pandas as pd, os, numpy as np

# Smart loader — local → GitHub → synthetic sample
LOCAL_PATH = 'datasets/tickets_labelled_5k.csv'
REMOTE_URL = 'https://raw.githubusercontent.com/internshipinabook/nlp-internship-in-a-book/main/data/tickets_labelled_5k.csv'

if os.path.exists(LOCAL_PATH):
    df = pd.read_csv(LOCAL_PATH)
    print('Loaded from local file ✅')
else:
    try:
        df = pd.read_csv(REMOTE_URL)
        print('Loaded from GitHub ✅')
    except Exception as e:
        print(f'⚠️  Generating synthetic fairness data ({e})')
        np.random.seed(42)
        n = 5000
        texts = ['customer support ticket text'] * n
        langs = np.random.choice(['en','pcm','yo','ha'], n, p=[0.55,0.25,0.12,0.08])
        cats  = np.random.choice(['billing','technical','account','general'], n)
        df = pd.DataFrame({
            'ticket_id':        range(n),
            'text':             texts,
            'text_clean':       texts,
            'category':         cats,
            'language_detected':langs,
        })
        print('Synthetic fairness dataset ready (5,000 rows) ✅')

X = df['text_clean'].fillna('').values
y = df['category'].values
print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head()


## Part 3 — Dialect Bias Gradient

In [ ]:
PIDGIN_MARKERS = {"abeg","una","don","dey","wahala","wetin","shey","abi","sha"}

test_df = pd.DataFrame({"text": X_test, "true": y_test, "pred": y_pred_afro})
test_df["pidgin_markers"] = test_df["text"].apply(
    lambda t: len(set(t.lower().split()) & PIDGIN_MARKERS))
test_df["correct"] = test_df["true"] == test_df["pred"]

print("Accuracy by Pidgin marker count:\n")
for n_markers in range(6):
    mask = test_df["pidgin_markers"] == n_markers
    n = mask.sum()
    if n < 5: continue
    acc = test_df.loc[mask, "correct"].mean()
    bar = "█" * int(acc * 20)
    print(f"  {n_markers} markers: {acc:.2f}  {bar}  (n={n})")

# Plot
valid_counts = [(nm, test_df[test_df["pidgin_markers"]==nm]["correct"].mean())
                for nm in range(6) if (test_df["pidgin_markers"]==nm).sum() >= 5]
if valid_counts:
    x, y_acc = zip(*valid_counts)
    fig, ax = plt.subplots(figsize=(8,4))
    ax.plot(x, y_acc, marker="o", color="#2E75B6", linewidth=2)
    ax.set_title("Model Accuracy Decreases as Pidgin Marker Count Increases",fontweight="bold",loc="left")
    ax.set_xlabel("Pidgin markers in ticket"); ax.set_ylabel("Accuracy")
    ax.axhline(0.75,color="#C0392B",linestyle="--",label="Deployment threshold")
    ax.legend(); ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    plt.tight_layout()
    plt.savefig("outputs/dialect_bias.png",dpi=150,bbox_inches="tight",facecolor="white")
    plt.show()

## Part 4 — Mitigation: Pidgin Routing Flag

In [ ]:
# Apply threshold adjustment for Pidgin-heavy tickets
PIDGIN_THRESHOLD = 2  # route to human review if 2+ Pidgin markers

test_df["is_pidgin_heavy"] = test_df["pidgin_markers"] >= PIDGIN_THRESHOLD
test_df["requires_review"] = test_df["is_pidgin_heavy"]

n_flagged = test_df["requires_review"].sum()
print(f"Tickets flagged for human review: {n_flagged} ({n_flagged/len(test_df):.1%})")

# Cost-benefit analysis
human_cost_per_ticket = 0.50  # USD
monthly_pidgin_tickets = 10000
monthly_cost = n_flagged / len(test_df) * monthly_pidgin_tickets * human_cost_per_ticket

abuse_caught_extra = 50  # estimated additional per month
harm_prevented_per = 200
monthly_value = abuse_caught_extra * harm_prevented_per

print(f"\nMonthly cost of Pidgin routing: ${monthly_cost:,.0f}")
print(f"Monthly value (additional abuse detected): ${monthly_value:,}")
print(f"Net benefit: ${monthly_value - monthly_cost:,.0f}")

## Part 5 — The Fairness Brief

> Write in this markdown cell:

## NLP Fairness Audit — CustomerCare Classifier v1.0

**Model audited:** AfroXLMR fine-tuned on 5,000 labelled tickets

### Findings
1. **Language gap:** English F1=[X.XXX] | Pidgin F1=[X.XXX] | Gap=[X] points ⚠️
2. **Dialect bias:** Accuracy decreases monotonically with Pidgin marker count
3. **Abuse intersection:** Pidgin abuse via social channel = lowest-performing subgroup

### What Was Done
- AfroXLMR (17 African languages) replacing English-only DistilBERT
- Pidgin oversampling (3×) during fine-tuning
- Pidgin-heavy tickets routed to human review

### Residual Disparity
A [N]-point gap remains. Root cause: insufficient Pidgin training data.
Commitment: 500 additional Pidgin labels in 60 days; immediate retraining.

### Recommendation
CONDITIONAL APPROVAL — deploy with Pidgin routing flag active.

📚 Reference: Blodgett et al. (2020) *Language (Technology) is Power*, ACL 2020
📚 Reference: Bender et al. (2021) *Stochastic Parrots*, FAccT 2021

## 🏆 Challenge Task

> Read Blodgett et al. (2020). Identify two claims about dialect bias you can verify with your corpus. Write a 150-word response in a markdown cell: 'In our corpus, claim X is [confirmed/partially confirmed/not observed] because...'

## ⚠️ Common Mistakes

| Mistake | What goes wrong | Fix |
|---------|-----------------|-----|
| Reporting only overall accuracy in the model card | A model that is accurate on average can systematically fail one customer group. Disaggregated metrics are not optional. | Always report precision, recall, and F1 per demographic or category group |
| Treating fairness as a binary pass/fail | There is no model that is perfectly fair across all metrics simultaneously. Fairness involves trade-offs that must be documented and communicated. | Document every disparity and its business implication — even the ones you cannot fix |

## ✅ What You Can Do After This Week

- Categorise NLP bias as representation, measurement, or aggregation
- Audit classifier performance across language groups
- Visualise dialect bias as a function of linguistic marker count
- Apply threshold-based fairness mitigation
- Write a structured fairness brief for legal review

---
## ✅ Week 10 Complete
**Next:** `week11_deploying_STARTER.ipynb`

---
*Add `week10_fairness_STARTER.ipynb` to your internship portfolio.*

*The NLP Internship · LinguaAI Labs · github.com/InternshipInABook*

## ✅ By completing Week 10, you can now:

- Compute fairness metrics disaggregated by language group
- Explain the fairness-accuracy trade-off in plain language to a non-technical stakeholder
- Write a fairness brief that a compliance team could use
- Propose and document a mitigation strategy with measured before/after impact

📤 **GitHub:** Push week10_fairness_STARTER.ipynb and fairness_brief.md. Commit: "Week 10: Fairness audit — dialect disparity documented"


---

## 📚 Get the Full Book

This notebook is part of **The NLP Internship** — Book 2 of the InternshipInABook™ Series.

The full book includes:
- Complete week-by-week narrative and explanations
- All STOP AND TRACE code walkthroughs
- Fairness briefs, model cards, and deployment guides
- Certificate of Completion

👉 **[Get the book on Selar →](https://selar.com/8440iqfr61)**

---
*InternshipInABook™ Series · © Sakinat Folorunso 2026 · [github.com/internshipinabook](https://github.com/internshipinabook)*
